# Tính tay và giải thích tham số

Ví dụ tự đặt giúp hiểu attention, sigmoid, BCE và gradient. Số minh họa không phải kết quả GoEmotions. Notebook không lập thêm thí nghiệm LR/epoch và không huấn luyện.

In [1]:
from pathlib import Path
import sys
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src').is_dir())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from examples.pretrained_math import calculate
calculate()

{'scope': 'two-label toy, not benchmark or pretrained inference',
 'attention_scores': [0.7071067811865475, 0.0],
 'attention_weights': [0.6697615493266569, 0.3302384506733431],
 'attention_output': [0.6697615493266569, 0.3302384506733431],
 'logits': [0.9, -0.4],
 'probabilities': [0.7109495026250039, 0.401312339887548],
 'mean_bce': 0.4270845635660202,
 'gradient_logits': [-0.14452524868749805, 0.200656169943774],
 'gradient_weight_gratitude': [-0.08671514921249883, 0.028905049737499612],
 'sgd_weight_new': [1.2000017343029843, -0.40000057810099476],
 'sgd_bias_new': 0.10000289050497375,
 'finite_difference_check_passed': True,
 'micro_batches_per_epoch': 2714}

## Cấu hình thực tế của ba kiến trúc
Learning rate và epoch là cấu hình đã chạy, chưa có ablation chứng minh tối ưu. Seed điều khiển khởi tạo head, dropout và shuffle; số 42, 123 hay 2026 không có ưu thế riêng.

In [2]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/notebook_support.py').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.notebook_support import artifacts_root, benchmark, replay_split
from src.models.transformer import run_folder, load_transformer_run, METRIC_NAMES
from src.evaluation.protocols import read_json
from src.evaluation.metrics import evaluate_multilabel
from src.datasets.goemotions import multi_hot
ARTIFACTS = artifacts_root()
verified, frames, labels = benchmark(str(ARTIFACTS))
SEEDS = (42, 123, 2026)
pd.set_option('display.max_columns', 20)
print('Python:', sys.version.split()[0])
print('Đã kiểm: 72 dòng kết quả và 9 checkpoint; ba seed cho mỗi C.')
configurations=[]
for a in ('bert','roberta','distilbert'):
    for s in SEEDS:
        m=load_transformer_run(run_folder(ARTIFACTS,a,s),require_full=True)
        configurations.append(dict(architecture=a,seed=s,selected_epoch=m['selected_epoch'],**m['config']))
display(pd.DataFrame(configurations)[['architecture','seed','learning_rate','epochs','batch_size','max_length','weighted','selected_epoch']])

PASS A: 12 dòng từ scores thật


PASS B: 6 dòng từ scores thật


PASS C: bert seed 42


PASS C: bert seed 123


PASS C: bert seed 2026


...

PASS C: distilbert seed 42


PASS C: distilbert seed 123


PASS C: distilbert seed 2026


Python: 3.13.9
Đã kiểm: 72 dòng kết quả và 9 checkpoint; ba seed cho mỗi C.


,architecture,seed,learning_rate,epochs,batch_size,max_length,weighted,selected_epoch
0,bert,42,0.00005,4,16,128,False,4
1,bert,123,0.00005,4,16,128,False,4
2,bert,2026,0.00005,4,16,128,False,4
3,roberta,42,0.00002,3,16,128,False,3
4,roberta,123,0.00002,3,16,128,False,3
5,roberta,2026,0.00002,3,16,128,False,3
6,distilbert,42,0.00002,3,16,128,False,3
7,distilbert,123,0.00002,3,16,128,False,3
8,distilbert,2026,0.00002,3,16,128,False,3


## Cách đọc chẩn đoán
Loss dao động giữa mini-batch chưa chứng minh mô hình thất bại. Cần đọc xu hướng train/validation, Macro-F1 và luật ngưỡng. Gap nhỏ có thể do cả train lẫn validation đều thấp; epoch lớn có thể làm quá khớp, LR quá nhỏ có thể học chưa đủ. Các nhận xét phải gắn với số đo, không kết luận nhân quả từ một checkpoint.

In [3]:
rows=[]
for a in ('bert','roberta','distilbert'):
    d=read_json(ROOT/f'results/final_submission/report_data/{a}_diagnostic.json')
    for r in d['rows']:rows.append(dict(architecture=a,seed=d['seed'],**r))
display(pd.DataFrame(rows))
print('Eval float32, ngưỡng 0,5; đây không phải loss trong lúc train.')

,architecture,seed,split,n,bce,macro_f1,micro_f1,hamming_loss
0,bert,123,train,43410,0.038038,0.727728,0.847516,0.011977
1,bert,123,validation,5426,0.091837,0.477356,0.583900,0.031811
2,roberta,2026,train,43410,0.069697,0.497949,0.660931,0.024381
3,roberta,2026,validation,5426,0.083689,0.430831,0.579166,0.029968
4,distilbert,123,train,43410,0.069196,0.478755,0.651177,0.024752
5,distilbert,123,validation,5426,0.084083,0.413200,0.573937,0.030021


Eval float32, ngưỡng 0,5; đây không phải loss trong lúc train.
